In [7]:
import requests
from bs4 import BeautifulSoup
import csv
from google.colab import files

# Function to extract property details from a property page
def extract_property_details(url):
    response = requests.get(url)
    html_content = response.content
    soup = BeautifulSoup(html_content, 'html.parser')

    property_details = {}

    # Extracting address details
    panel_body = soup.find('div', class_='panel-body')
    if panel_body:
        listing_details = panel_body.find_all('div', class_='listing_detail')
        for detail in listing_details:
            strong_tag = detail.find('strong')
            if strong_tag:
                key = strong_tag.get_text(strip=True).replace(':', '')
                value = detail.get_text(strip=True).replace(key + ':', '').strip()
                property_details[key] = value

    # Extracting leasing brokers
    office_inquiries = soup.find('div', class_='office-inquiries')
    leasing_brokers = []
    if office_inquiries:
        brokers = office_inquiries.find_all('h5')
        for broker in brokers:
            broker_name = broker.get_text(strip=True)
            broker_phone = broker.find_next_sibling('p').contents[0].strip().replace('Phone: ', '')
            broker_email = broker.find_next_sibling('p').find('a').get_text(strip=True)
            leasing_brokers.append(f"{broker_name}, {broker_phone}, {broker_email}")
    property_details['Leasing Brokers'] = ' | '.join(leasing_brokers)

    return property_details

# Main URL for CCI Owned Properties
main_url = 'https://www.capitalcommercial.com/cci-owned-properties/'

# Make a GET request to fetch the raw HTML content of the main page
main_response = requests.get(main_url)
main_html_content = main_response.content
main_soup = BeautifulSoup(main_html_content, 'html.parser')

# Find all property links
property_listings = main_soup.find_all('div', class_='property_listing')
property_urls = [listing.find('a', href=True)['href'] for listing in property_listings]

# Print fetched URLs to verify
print("Fetched URLs:")
for url in property_urls:
    print(url)

# List to store all property details
all_property_details = []

# Loop through each property link and extract details
for property_url in property_urls:
    property_details = extract_property_details(property_url)
    all_property_details.append(property_details)

# Define CSV file path
csv_file_path = 'all_property_details.csv'

# Write all property details to CSV file
with open(csv_file_path, mode='w', newline='', encoding='utf-8') as csv_file:
    fieldnames = ['Address', 'City', 'State/County', 'Zip', 'Country', 'Leasing Brokers']
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)

    writer.writeheader()
    for details in all_property_details:
        writer.writerow({
            'Address': details.get('Address', ''),
            'City': details.get('City', ''),
            'State/County': details.get('State/County', ''),
            'Zip': details.get('Zip', ''),
            'Country': details.get('Country', ''),
            'Leasing Brokers': details.get('Leasing Brokers', '')
        })

print(f"All property details saved to {csv_file_path}")

# Download the CSV file in Google Colab
files.download(csv_file_path)


Fetched URLs:
https://www.capitalcommercial.com/city/tulum/
https://www.capitalcommercial.com/city/irving/
https://www.capitalcommercial.com/city/houston/
https://www.capitalcommercial.com/city/irving/
https://www.capitalcommercial.com/properties/11125-11025-equity-drive/
https://www.capitalcommercial.com/area/galleria-uptown/
https://www.capitalcommercial.com/area/houston/
https://www.capitalcommercial.com/area/houston/
https://www.capitalcommercial.com/area/bellaire/
https://www.capitalcommercial.com/area/northwest/
https://www.capitalcommercial.com/area/north-hwy-tomball-pky-ind/
https://www.capitalcommercial.com/area/north-hardy-toll-road-ind/
All property details saved to all_property_details.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
from google.colab import files

# URL of the property page
url = 'https://www.capitalcommercial.com/properties/tulum-hotel-cabanas-la-luna-beach-restaurant/'

# Make a GET request to fetch the raw HTML content
response = requests.get(url)
html_content = response.content

# Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# Extract property details
property_details = {}

# Extracting address details
panel_body = soup.find('div', class_='panel-body')
if panel_body:
    listing_details = panel_body.find_all('div', class_='listing_detail')
    for detail in listing_details:
        strong_tag = detail.find('strong')
        if strong_tag:
            key = strong_tag.get_text(strip=True).replace(':', '')
            value = strong_tag.next_sibling.strip() if strong_tag.next_sibling else ""
            if not value and strong_tag.next_sibling and strong_tag.next_sibling.name == 'a':
                value = strong_tag.next_sibling.get_text(strip=True)
            property_details[key] = value

# Extracting leasing brokers
office_inquiries = soup.find('div', class_='office-inquiries')
leasing_brokers = []
if office_inquiries:
    brokers = office_inquiries.find_all('h5')
    for broker in brokers:
        broker_name = broker.get_text(strip=True)
        broker_phone = broker.find_next_sibling('p').contents[0].strip().replace('Phone: ', '')
        broker_email = broker.find_next_sibling('p').find('a').get_text(strip=True)
        leasing_brokers.append(f"{broker_name}, {broker_phone}, {broker_email}")
property_details['Leasing Brokers'] = ' | '.join(leasing_brokers)

# Define CSV file path
csv_file_path = 'property_details.csv'

# Write property details to CSV file
with open(csv_file_path, mode='w', newline='', encoding='utf-8') as csv_file:
    fieldnames = ['Address', 'City', 'State/County', 'Zip', 'Country', 'Leasing Brokers']
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)

    writer.writeheader()
    writer.writerow({
        'Address': property_details.get('Address', ''),
        'City': property_details.get('City', ''),
        'State/County': property_details.get('State/County', ''),
        'Zip': property_details.get('Zip', ''),
        'Country': property_details.get('Country', ''),
        'Leasing Brokers': property_details.get('Leasing Brokers', '')
    })

print(f"Property details saved to {csv_file_path}")

# Download the CSV file in Google Colab
files.download(csv_file_path)


In [17]:
import requests
from bs4 import BeautifulSoup
import csv
from google.colab import files
import re

def extract_property_details(url):
    response = requests.get(url)
    html_content = response.content
    soup = BeautifulSoup(html_content, 'html.parser')

    property_details = {}

    # Extracting address details
    panel_body = soup.find('div', class_='panel-body')
    if panel_body:
        listing_details = panel_body.find_all('div', class_='listing_detail')
        for detail in listing_details:
            strong_tag = detail.find('strong')
            if strong_tag:
                key = strong_tag.get_text(strip=True).replace(':', '')
                value = detail.get_text(strip=True).replace(key + ':', '').strip()
                property_details[key] = value

    # Extracting leasing brokers
    textwidget_div = soup.find('div', class_='textwidget')
    leasing_brokers = []
    if textwidget_div:
        broker_paragraphs = textwidget_div.find_all('p')
        for p in broker_paragraphs:
            broker_info = {}
            # Extract Name
            name_tag = p.find(text=re.compile(r'^Name:'))
            if name_tag:
                broker_name = name_tag.split(':', 1)[1].strip()
            else:
                broker_name = 'N/A'

            # Extract Phone
            phone_tag = p.find(text=re.compile(r'^Phone:'))
            if phone_tag:
                broker_phone = phone_tag.split(':', 1)[1].strip()
            else:
                broker_phone = 'N/A'

            # Extract Email
            email_tag = p.find('a', href=True)
            broker_email = email_tag.get_text(strip=True) if email_tag else 'N/A'

            leasing_brokers.append(f"{broker_name}, {broker_phone}, {broker_email}")

    property_details['Leasing Brokers'] = ' | '.join(leasing_brokers)

    return property_details





# Main URL for CCI Owned Properties
main_url = 'https://www.capitalcommercial.com/cci-owned-properties/'

# Make a GET request to fetch the raw HTML content of the main page
main_response = requests.get(main_url)
main_html_content = main_response.content
main_soup = BeautifulSoup(main_html_content, 'html.parser')

# Find all property links

property_listings = main_soup.find_all('div', class_='property-unit-information-wrapper')
property_urls = [
    listing.find('a', href=True)['href']
    for listing in property_listings
    if re.match(r'^https://www\.capitalcommercial\.com/properties/.*/$', listing.find('a', href=True)['href'])
]


# Print fetched URLs to verify
print("Fetched URLs:")
for url in property_urls:
    print(url)

# List to store all property details
all_property_details = []

# Loop through each property link and extract details
for property_url in property_urls:
    property_details = extract_property_details(property_url)
    all_property_details.append(property_details)

# Define CSV file path
csv_file_path = 'all_property_details.csv'

# Write all property details to CSV file
with open(csv_file_path, mode='w', newline='', encoding='utf-8') as csv_file:
    fieldnames = ['Address', 'City', 'State/County', 'Zip', 'Country', 'Leasing Brokers']
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)

    writer.writeheader()
    for details in all_property_details:
        writer.writerow({
            'Address': details.get('Address', ''),
            'City': details.get('City', ''),
            'State/County': details.get('State/County', ''),
            'Zip': details.get('Zip', ''),
            'Country': details.get('Country', ''),
            'Leasing Brokers': details.get('Leasing Brokers', '')
        })

print(f"All property details saved to {csv_file_path}")

# Download the CSV file in Google Colab
files.download(csv_file_path)

Fetched URLs:
https://www.capitalcommercial.com/properties/tulum-hotel-cabanas-la-luna-beach-restaurant/
https://www.capitalcommercial.com/properties/5959-las-colinas-blvd/
https://www.capitalcommercial.com/properties/17404-katy-freeway/
https://www.capitalcommercial.com/properties/4200-regent-blvd/
https://www.capitalcommercial.com/properties/11125-11025-equity-drive/
https://www.capitalcommercial.com/properties/1900-west-loop-south/
https://www.capitalcommercial.com/properties/11210-equity-drive/
https://www.capitalcommercial.com/properties/10777-clay-rd/
https://www.capitalcommercial.com/properties/6500-west-loop-south/
https://www.capitalcommercial.com/properties/11403-compaq/
https://www.capitalcommercial.com/properties/6615-gant-road/
https://www.capitalcommercial.com/properties/5405-consulate-plaza-drive/
All property details saved to all_property_details.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>